# Custom Vision 객체 감지 프로젝트

In [32]:
# 클라이언트 라이브러리 설치
%pip install azure-cognitiveservices-vision-customvision

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 라이브러리 및 기본 정보 설정

In [33]:
#========== 라이브러리 ==========
from azure.cognitiveservices.vision.customvision.training import CustomVisionTrainingClient
from azure.cognitiveservices.vision.customvision.prediction import CustomVisionPredictionClient
from azure.cognitiveservices.vision.customvision.training.models import ImageFileCreateBatch, ImageFileCreateEntry, Region
from msrest.authentication import ApiKeyCredentials
import os, time, uuid, dotenv

In [34]:
#========== 엔드포인트 및 키 ==========
dotenv.load_dotenv()
CV_TRAINING_ENDPOINT = os.getenv("CV_TRAINING_ENDPOINT")
CV_TRAINING_KEY = os.getenv("CV_TRAINING_KEY")
CV_PREDICTION_ENDPOINT = os.getenv("CV_PREDICTION_ENDPOINT")
CV_PREDICTION_KEY = os.getenv("CV_PREDICTION_KEY")
CV_PREDICTION_RESOURCE_ID = os.getenv("CV_PREDICTION_RESOURCE_ID")

TRAINING_ENDPOINT = CV_TRAINING_ENDPOINT
PREDICTION_ENDPOINT = CV_PREDICTION_ENDPOINT

TRAINING_API_KEY = CV_TRAINING_KEY
PREDICTION_API_KEY = CV_PREDICTION_KEY

PREDICTION_RESOURCE_ID = CV_PREDICTION_RESOURCE_ID

## Trainer, Predictor 객체화

In [35]:
#========== SDK를 이용해 클라이언트 초기화 ==========

# ApiKeyCredentials 객체에 인증 정보를 한 번 담으면
# 이후 trainer 객체가 하는 모든 HTTP 요청에 자동으로 헤더를 붙여줍니다.
# Raw HTTP였다면 아래 나오는 모든 requests.post() 호출마다 headers=를 반복해야 합니다.
training_credentials = ApiKeyCredentials(in_headers={"Training-key": TRAINING_API_KEY})
prediction_credentials = ApiKeyCredentials(in_headers={"Prediction-key": PREDICTION_API_KEY})

trainer = CustomVisionTrainingClient(endpoint=TRAINING_ENDPOINT, credentials=training_credentials)
predictor = CustomVisionPredictionClient(endpoint=PREDICTION_ENDPOINT, credentials=prediction_credentials)

## 프로젝트 만들기

### 프로젝트 및 도메인 정보 확인

In [36]:
#========== 프로젝트 확인 ==========
# 생성한 프로젝트가 없다면 뜨지 않음 
project_list = trainer.get_projects()

for project in project_list:
    print(
        "Project Name: {}".format(project.name),
        '|',
        "Project ID: {}".format(project.id)
    )
#========== 도메인 확인 ==========
# 프로젝트 생성시 어떤 도메인이 존재하는지 확인 
# 이미지를 어떤 유형으로 햑습시킬지 선택
# 업로드된 이미지가 특정 도메인으로 분류될 시 제외할 수 있음
domain_list = trainer.get_domains()

for domain in domain_list:
    print(
        "Domain Type: {}\n".format(domain.type),
        "Domain Name: {}\n".format(domain.name),
        "Domain ID: {}\n".format(domain.id),
    )

Project Name: 9ai043-kitchen | Project ID: fe261064-8dff-4926-b28c-ff38b9714989
Domain Type: Classification
 Domain Name: General [A2]
 Domain ID: 2e37d7fb-3a54-486a-b4d6-cfc369af0018

Domain Type: Classification
 Domain Name: General [A1]
 Domain ID: a8e3c40f-fb4a-466f-832a-5e457ae4a344

Domain Type: Classification
 Domain Name: General
 Domain ID: ee85a74c-405e-4adc-bb47-ffa8ca0c9f31

Domain Type: Classification
 Domain Name: Food
 Domain ID: c151d5b5-dd07-472a-acc8-15d29dea8518

Domain Type: Classification
 Domain Name: Landmarks
 Domain ID: ca455789-012d-4b50-9fec-5bb63841c793

Domain Type: Classification
 Domain Name: Retail
 Domain ID: b30a91ae-e3c1-4f73-a81e-c270bff27c39

Domain Type: Classification
 Domain Name: Adult
 Domain ID: 45badf75-3591-4f26-a705-45678d3e9f5f

Domain Type: Classification
 Domain Name: General (compact) [S1]
 Domain ID: a1db07ca-a19a-4830-bae8-e004a42dc863

Domain Type: Classification
 Domain Name: General (compact)
 Domain ID: 0732100f-1a38-4e49-a514-c9b

### 새로운 프로젝트 만들기

In [37]:
#========== 새로운 프로젝트 정보 설정 ==========
project_name = "9ai043-kitchen"
project_description = "포크와 가위를 감지하는 모델"

#========== 프로젝트 중복 확인 ==========
# 이름이 중복되어도 생성은 되지만, 이름(2), 이름(3) 이런 식으로 생성되므로 
# 중복 확인을 사전에 하는 게 좋음 
project_list = trainer.get_projects()
project = None
domain=None

for proj in project_list:
    if project_name == proj.name:
        print('이미 존재하는 프로젝트를 불러옵니다.')
        project = proj
        break

#========== 프로젝트 생성 ==========
# 프로젝트가 존재하지 않을 경우, 
if project is None:
    print('새로운 프로젝트를 생성합니다.')

    #========== 원하는 도메인 찾기 ==========
    for d in domain_list:
        if d.type == "ObjectDetection" and d.name == "General (compact)":
            domain = d
            print(f'도메인 {d.type, d.name, d.id}을(를) 선택합니다.')
            break
    
    #========== 프로젝트 생성 ==========
    project = trainer.create_project(name=project_name, description=project_description, domain_id=domain.id)
    print('프로젝트가 생성되었습니다.')


이미 존재하는 프로젝트를 불러옵니다.


In [38]:
project.id

'fe261064-8dff-4926-b28c-ff38b9714989'

### 태그 추가하기

In [39]:
#========== 태그 설정 ==========
FORK_TAG_NAME = "포크"
SCISSORS_TAG_NAME = "가위"

fork_tag = None
scissors_tag = None

#========== 존재하는 태그 불러오기 ==========
# 가장 최근에 생성된 태그가 0번째로 들어감 
tag_list = trainer.get_tags(project.id)
for tag in tag_list:
    if tag.name == FORK_TAG_NAME:
        print('이미 존재하는 포크 태그를 불러옵니다.')
        fork_tag = tag
    if tag.name == SCISSORS_TAG_NAME:
        print('이미 존재하는 가위 태그를 불러옵니다.')
        scissors_tag = tag

#========== 존재하는 태그가 없을 때만 태그 생성하기 ==========
# 학습시킬 이미지에 태그를 달아야 하는데, 태그 아이디를 알아야 함
# 그래서 fork_tag, scissors_tag에 저장해두는 것
if fork_tag is None:
    fork_tag =trainer.create_tag(project.id, FORK_TAG_NAME)
    print(f'포크 태그({fork_tag.id})가 생성되었습니다.')
if scissors_tag is None:
    scissors_tag = trainer.create_tag(project.id, SCISSORS_TAG_NAME)
    print(f'가위 태그({scissors_tag.id})가 생성되었습니다.')

이미 존재하는 포크 태그를 불러옵니다.
이미 존재하는 가위 태그를 불러옵니다.


In [40]:
# 확인해보기
trainer.get_tags(project.id)

## 이미지 업로드 및 태그 지정

### 태그 지정
* 0~1 사이 값으로 정규화된 좌표
* ( x좌표, y좌표, w넓이, h높이 )
* 각 좌표들은 객체화 필요

In [41]:
#========== 포크 이미지 바운딩 박스 좌표 지정 ==========
fork_image_regions = {
    "fork_1": [0.145833328, 0.3509314, 0.5894608, 0.238562092],
    "fork_2": [0.294117659, 0.216944471, 0.534313738, 0.5980392],
    "fork_3": [0.09191177, 0.0682516545, 0.757352948, 0.6143791],
    "fork_4": [0.254901975, 0.185898721, 0.5232843, 0.594771266],
    "fork_5": [0.2365196, 0.128709182, 0.5845588, 0.71405226],
    "fork_6": [0.115196079, 0.133611143, 0.676470637, 0.6993464],
    "fork_7": [0.164215669, 0.31008172, 0.767156839, 0.410130739],
    "fork_8": [0.118872553, 0.318251669, 0.817401946, 0.225490168],
    "fork_9": [0.18259804, 0.2136765, 0.6335784, 0.643790841],
    "fork_10": [0.05269608, 0.282303959, 0.8088235, 0.452614367],
    "fork_11": [0.05759804, 0.0894935, 0.9007353, 0.3251634],
    "fork_12": [0.3345588, 0.07315363, 0.375, 0.9150327],
    "fork_13": [0.269607842, 0.194068655, 0.4093137, 0.6732026],
    "fork_14": [0.143382356, 0.218578458, 0.7977941, 0.295751631],
    "fork_15": [0.19240196, 0.0633497, 0.5710784, 0.8398692],
    "fork_16": [0.140931368, 0.480016381, 0.6838235, 0.240196079],
    "fork_17": [0.305147052, 0.2512582, 0.4791667, 0.5408496],
    "fork_18": [0.234068632, 0.445702642, 0.6127451, 0.344771236],
    "fork_19": [0.219362751, 0.141781077, 0.5919118, 0.6683006],
    "fork_20": [0.180147052, 0.239820287, 0.6887255, 0.235294119],
}

#========== 가위 이미지 바운딩 박스 좌표 지정 ==========
scissors_image_regions = {
    "scissors_1": [0.4007353, 0.194068655, 0.259803921, 0.6617647],
    "scissors_2": [0.426470578, 0.185898721, 0.172794119, 0.5539216],
    "scissors_3": [0.289215684, 0.259428144, 0.403186262, 0.421568632],
    "scissors_4": [0.343137264, 0.105833367, 0.332107842, 0.8055556],
    "scissors_5": [0.3125, 0.09766343, 0.435049027, 0.71405226],
    "scissors_6": [0.379901975, 0.24308826, 0.32107842, 0.5718954],
    "scissors_7": [0.341911763, 0.20714055, 0.3137255, 0.6356209],
    "scissors_8": [0.231617644, 0.08459154, 0.504901946, 0.8480392],
    "scissors_9": [0.170343131, 0.332957536, 0.767156839, 0.403594762],
    "scissors_10": [0.204656869, 0.120539248, 0.5245098, 0.743464053],
    "scissors_11": [0.05514706, 0.159754932, 0.799019635, 0.730392158],
    "scissors_12": [0.265931368, 0.169558853, 0.5061275, 0.606209159],
    "scissors_13": [0.241421565, 0.184264734, 0.448529422, 0.6830065],
    "scissors_14": [0.05759804, 0.05027781, 0.75, 0.882352948],
    "scissors_15": [0.191176474, 0.169558853, 0.6936275, 0.6748366],
    "scissors_16": [0.1004902, 0.279036, 0.6911765, 0.477124184],
    "scissors_17": [0.2720588, 0.131977156, 0.4987745, 0.6911765],
    "scissors_18": [0.180147052, 0.112369314, 0.6262255, 0.6666667],
    "scissors_19": [0.333333343, 0.0274019931, 0.443627447, 0.852941155],
    "scissors_20": [0.158088237, 0.04047389, 0.6691176, 0.843137264],
}

### 이미지 파일 불러오기 및 업로드

In [42]:
#========== 이미지 파일 경로 테스트 ==========
with open("./cv-image/fork/fork_1.jpg", "rb") as f:
    image_data = f.read()
    print(f'fork: {image_data}')

with open("./cv-image/scissors/scissors_1.jpg", "rb") as f:
    image_data = f.read()
    print(f'scissors: {image_data}')

fork: b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x01\x00`\x00`\x00\x00\xff\xdb\x00C\x00\x08\x06\x06\x07\x06\x05\x08\x07\x07\x07\t\t\x08\n\x0c\x14\r\x0c\x0b\x0b\x0c\x19\x12\x13\x0f\x14\x1d\x1a\x1f\x1e\x1d\x1a\x1c\x1c $.\' ",#\x1c\x1c(7),01444\x1f\'9=82<.342\xff\xdb\x00C\x01\t\t\t\x0c\x0b\x0c\x18\r\r\x182!\x1c!22222222222222222222222222222222222222222222222222\xff\xc0\x00\x11\x08\x02d\x030\x03\x01"\x00\x02\x11\x01\x03\x11\x01\xff\xc4\x00\x1f\x00\x00\x01\x05\x01\x01\x01\x01\x01\x01\x00\x00\x00\x00\x00\x00\x00\x00\x01\x02\x03\x04\x05\x06\x07\x08\t\n\x0b\xff\xc4\x00\xb5\x10\x00\x02\x01\x03\x03\x02\x04\x03\x05\x05\x04\x04\x00\x00\x01}\x01\x02\x03\x00\x04\x11\x05\x12!1A\x06\x13Qa\x07"q\x142\x81\x91\xa1\x08#B\xb1\xc1\x15R\xd1\xf0$3br\x82\t\n\x16\x17\x18\x19\x1a%&\'()*456789:CDEFGHIJSTUVWXYZcdefghijstuvwxyz\x83\x84\x85\x86\x87\x88\x89\x8a\x92\x93\x94\x95\x96\x97\x98\x99\x9a\xa2\xa3\xa4\xa5\xa6\xa7\xa8\xa9\xaa\xb2\xb3\xb4\xb5\xb6\xb7\xb8\xb9\xba\xc2\xc3\xc4\xc5\xc6\xc7\xc8\xc9\xca\xd2\xd3\xd4\xd

In [43]:
#========== 이미지 + 이미지에 해당하는 좌표 불러오기 ==========
for file_name, bounding_box in fork_image_regions.items():
    with open(f"./cv-image/fork/{file_name}.jpg", "rb") as f:
        image_data = f.read()
        print(f'fork: {image_data}, bounding_box: {bounding_box}')


for file_name, bounding_box in scissors_image_regions.items():
    with open(f"./cv-image/scissors/{file_name}.jpg", "rb") as f:
        image_data = f.read()
        print(f'scissors: {image_data}, bounding_box: {bounding_box}')

fork: b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x01\x00`\x00`\x00\x00\xff\xdb\x00C\x00\x08\x06\x06\x07\x06\x05\x08\x07\x07\x07\t\t\x08\n\x0c\x14\r\x0c\x0b\x0b\x0c\x19\x12\x13\x0f\x14\x1d\x1a\x1f\x1e\x1d\x1a\x1c\x1c $.\' ",#\x1c\x1c(7),01444\x1f\'9=82<.342\xff\xdb\x00C\x01\t\t\t\x0c\x0b\x0c\x18\r\r\x182!\x1c!22222222222222222222222222222222222222222222222222\xff\xc0\x00\x11\x08\x02d\x030\x03\x01"\x00\x02\x11\x01\x03\x11\x01\xff\xc4\x00\x1f\x00\x00\x01\x05\x01\x01\x01\x01\x01\x01\x00\x00\x00\x00\x00\x00\x00\x00\x01\x02\x03\x04\x05\x06\x07\x08\t\n\x0b\xff\xc4\x00\xb5\x10\x00\x02\x01\x03\x03\x02\x04\x03\x05\x05\x04\x04\x00\x00\x01}\x01\x02\x03\x00\x04\x11\x05\x12!1A\x06\x13Qa\x07"q\x142\x81\x91\xa1\x08#B\xb1\xc1\x15R\xd1\xf0$3br\x82\t\n\x16\x17\x18\x19\x1a%&\'()*456789:CDEFGHIJSTUVWXYZcdefghijstuvwxyz\x83\x84\x85\x86\x87\x88\x89\x8a\x92\x93\x94\x95\x96\x97\x98\x99\x9a\xa2\xa3\xa4\xa5\xa6\xa7\xa8\xa9\xaa\xb2\xb3\xb4\xb5\xb6\xb7\xb8\xb9\xba\xc2\xc3\xc4\xc5\xc6\xc7\xc8\xc9\xca\xd2\xd3\xd4\xd

### 이미지 업로드

In [ ]:
#========== 이미지 + 이미지에 해당하는 좌표 불러오기 ==========
image_list = list()

#==========포크 이미지 데이터 업로드 ==========
for file_name, bounding_box in fork_image_regions.items():
    with open(f"./cv-image/fork/{file_name}.jpg", "rb") as f:
        image_data = f.read()
        left, top, width, height = bounding_box
        # 바운딩 박스 -> Region 객체로 전환 (바운딩 박스 태그 + 좌표)
        region = Region(
            tag_id = fork_tag.id,
            left=left,
            top=top,
            width=width,
            height=height
        )
        regions = [region]
        
        # image file 엔트리를 
        image_list.append(ImageFileCreateEntry(name=file_name, contents=image_data, regions=regions))

#==========가위 이미지 데이터 업로드 ==========
for file_name, bounding_box in scissors_image_regions.items():
    with open(f"./cv-image/scissors/{file_name}.jpg", "rb") as f:
        image_data = f.read()
        left, top, width, height = bounding_box
        print(f'scissors: {image_data}, bounding_box: {bounding_box}')
        region = Region(
            tag_id = scissors_tag.id,
            left=left,
            top=top,
            width=width,
            height=height
        )
        regions = [region]
        # image file 엔트리로 묶기 
        image_list.append(ImageFileCreateEntry(name=file_name, contents=image_data, regions=regions))

# 배치로 묶어서 이미지 업로드
# 한 번의 API 호출로 여러 이미지 업로드 
batch = ImageFileCreateBatch(images=image_list)

# 프로젝트에 이미지 생성(업로드)
result = trainer.create_images_from_files(project.id, batch)

for image in result.images:
    print("{} : {}".format(image.source_url, image.status))

scissors: b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x01\x00`\x00`\x00\x00\xff\xdb\x00C\x00\x08\x06\x06\x07\x06\x05\x08\x07\x07\x07\t\t\x08\n\x0c\x14\r\x0c\x0b\x0b\x0c\x19\x12\x13\x0f\x14\x1d\x1a\x1f\x1e\x1d\x1a\x1c\x1c $.\' ",#\x1c\x1c(7),01444\x1f\'9=82<.342\xff\xdb\x00C\x01\t\t\t\x0c\x0b\x0c\x18\r\r\x182!\x1c!22222222222222222222222222222222222222222222222222\xff\xc0\x00\x11\x08\x02d\x030\x03\x01"\x00\x02\x11\x01\x03\x11\x01\xff\xc4\x00\x1f\x00\x00\x01\x05\x01\x01\x01\x01\x01\x01\x00\x00\x00\x00\x00\x00\x00\x00\x01\x02\x03\x04\x05\x06\x07\x08\t\n\x0b\xff\xc4\x00\xb5\x10\x00\x02\x01\x03\x03\x02\x04\x03\x05\x05\x04\x04\x00\x00\x01}\x01\x02\x03\x00\x04\x11\x05\x12!1A\x06\x13Qa\x07"q\x142\x81\x91\xa1\x08#B\xb1\xc1\x15R\xd1\xf0$3br\x82\t\n\x16\x17\x18\x19\x1a%&\'()*456789:CDEFGHIJSTUVWXYZcdefghijstuvwxyz\x83\x84\x85\x86\x87\x88\x89\x8a\x92\x93\x94\x95\x96\x97\x98\x99\x9a\xa2\xa3\xa4\xa5\xa6\xa7\xa8\xa9\xaa\xb2\xb3\xb4\xb5\xb6\xb7\xb8\xb9\xba\xc2\xc3\xc4\xc5\xc6\xc7\xc8\xc9\xca\xd2\xd3\xd